In [1]:
# !wget https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_books.json.gz
# !wget https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_book_genres_initial.json.gz

--2025-11-23 21:40:58--  https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_books.json.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 169.228.63.88
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|169.228.63.88|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2083197934 (1.9G) [application/gzip]
Saving to: ‘goodreads_books.json.gz’

goodreads_books.jso  37%[======>             ] 754.63M  6.65MB/s    eta 3m 39s ^C


In [2]:
import pandas as pd

chunksize = 100_000  # 100만 행 단위로
chunks = pd.read_json(
    "goodreads_books.json.gz",
    lines=True,
    compression="gzip",
    chunksize=chunksize
)

In [2]:
# # goodreads_books.json.gz
# book_info = {'isbn': '1591935857',
#              'description': '',
#              'authors': [
#                  {'author_id': '13195', 'role': ''},
#                  {'author_id': '30853', 'role': 'Photographs'}
#              ],
#              'publisher': 'Adventurekeen',
#              'book_id': '27036533',
#              'title': 'Jump, Little Wood Ducks'}

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

out_path = "book_meta.parquet"
writer = None

for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i}...")

    chunk = chunk.loc[:, ['isbn', 'description', 'authors', 'publisher', 'book_id', 'title']].copy()

    # Arrow Table로 변환
    table = pa.Table.from_pandas(chunk)

    if writer is None:
        writer = pq.ParquetWriter(out_path, table.schema, compression='snappy')

    writer.write_table(table)

if writer:
    writer.close()

print("✅ Done! book_meta.parquet saved.")

In [1]:
import pandas as pd

chunksize = 100_000  # 100만 행 단위로
chunks = pd.read_json(
    # "goodreads_books.json.gz",
    "goodreads_book_genres_initial.json.gz",
    lines=True,
    compression="gzip",
    chunksize=chunksize
)

# 모든 chunk 합치기
df_genres = pd.concat(chunks, ignore_index=True)

print(type(df_genres['genres'].iloc[0]))

<class 'dict'>


In [2]:
import ast

# -----------------------------
# top 3 genre 선택
# -----------------------------
def top_n_genres(d, n=3):
    # dict items를 value 기준 내림차순 정렬 후 top n 선택
    sorted_items = sorted(d.items(), key=lambda x: x[1], reverse=True)
    # genre 이름만 리스트로 반환
    return [genre for genre, count in sorted_items[:n]]

df_genres['top_genres'] = df_genres['genres'].apply(lambda x: top_n_genres(x, 3))

print(df_genres[['book_id', 'top_genres']])

          book_id                                         top_genres
0         5333265           [history, historical fiction, biography]
1         1333909  [fiction, history, historical fiction, biography]
2         7327624  [fantasy, paranormal, fiction, mystery, thrill...
3         6066819       [fiction, romance, mystery, thriller, crime]
4          287140                                      [non-fiction]
...           ...                                                ...
2360650   3084038  [history, historical fiction, biography, non-f...
2360651  26168430      [mystery, thriller, crime, children, fiction]
2360652   2342551                    [poetry, children, young-adult]
2360653  22017381                [romance, mystery, thriller, crime]
2360654  11419866                                 [romance, fiction]

[2360655 rows x 2 columns]


In [3]:
DATA_PATH = 'book_meta.parquet'
df_books = pd.read_parquet(DATA_PATH)

In [4]:
df_merged = df_books.merge(df_genres, on='book_id', how='left')

In [5]:
print(df_merged.head())

         isbn                                        description  \
0  9770252646  A Tintin Film Book\nTintin, Snowy and Captain ...   
1  1421808420  Purchase one of 1st World Library's Classic Bo...   
2  0807032735  After graduating from Yale University, Sarah S...   
3  0321750403  "Learning iPad Programming" walks you through ...   
4              O novo livro de Pedro Paixao trata de uma curi...   

                                  authors  \
0  [{'author_id': '2802356', 'role': ''}]   
1     [{'author_id': '8164', 'role': ''}]   
2   [{'author_id': '250484', 'role': ''}]   
3  [{'author_id': '4767184', 'role': ''}]   
4  [{'author_id': '1385066', 'role': ''}]   

                              publisher   book_id  \
0                              dr lm`rf   6474616   
1  1st World Library - Literary Society    445501   
2                          Beacon Press    445503   
3           Addison-Wesley Professional  11007713   
4                           Prime Books  12636575   

 

In [6]:
print(type(df_merged['authors'].iloc[0]))

<class 'numpy.ndarray'>


In [7]:
import numpy as np

DATA_PATH = 'book_meta.parquet'
df_books = pd.read_parquet(DATA_PATH)
df_merged = df_books.copy()

def get_first_author_id(x):
    if isinstance(x, (list, np.ndarray)) and len(x) > 0:
        first = x[0]
        if isinstance(first, dict) and 'author_id' in first:
            return first['author_id']
    return None

df_merged['author_id'] = df_merged['authors'].apply(get_first_author_id)

print(df_merged[['book_id', 'author_id']].head())

    book_id author_id
0   6474616   2802356
1    445501      8164
2    445503    250484
3  11007713   4767184
4  12636575   1385066


In [8]:
cols_to_drop = ['isbn','authors', 'publisher', 'genres']  # 제거할 컬럼 이름
df_merged = df_merged.drop(columns=cols_to_drop, errors='ignore')  # errors='ignore' → 없는 컬럼이어도 무시

In [9]:
print(df_merged.head())

                                         description   book_id  \
0  A Tintin Film Book\nTintin, Snowy and Captain ...   6474616   
1  Purchase one of 1st World Library's Classic Bo...    445501   
2  After graduating from Yale University, Sarah S...    445503   
3  "Learning iPad Programming" walks you through ...  11007713   
4  O novo livro de Pedro Paixao trata de uma curi...  12636575   

                                               title  \
0                           تان تان والبحيرة الغامضة   
1                   Alice's Adventures in Wonderland   
2  Taught by America: A Story of Struggle and Hop...   
3  Learning iPad Programming: A Hands-On Guide to...   
4                                  A Rapariga Errada   

                                          top_genres author_id  
0               [comics, graphic, fiction, children]   2802356  
1           [children, fantasy, paranormal, fiction]      8164  
2  [non-fiction, history, historical fiction, bio...    250484  
3     

In [11]:
df_merged = df_merged.rename(columns={
    'top_genres': 'genres',
    # 필요하면 다른 컬럼도 여기서 변경 가능
})

In [12]:
# 원하는 컬럼 순서
cols_order = ['book_id', 'author_id', 'title', 'genres', 'description']

# 컬럼 순서 재정렬
df_merged = df_merged[cols_order]

In [13]:
df_merged

,book_id,author_id,title,genres,description
0,6474616,2802356,تان تان والبحيرة الغامضة,"[comics, graphic, fiction, children]","A Tintin Film Book\nTintin, Snowy and Captain ..."
1,445501,8164,Alice's Adventures in Wonderland,"[children, fantasy, paranormal, fiction]",Purchase one of 1st World Library's Classic Bo...
2,445503,250484,Taught by America: A Story of Struggle and Hop...,"[non-fiction, history, historical fiction, bio...","After graduating from Yale University, Sarah S..."
3,11007713,4767184,Learning iPad Programming: A Hands-On Guide to...,[non-fiction],"""Learning iPad Programming"" walks you through ..."
4,12636575,1385066,A Rapariga Errada,[],O novo livro de Pedro Paixao trata de uma curi...
...,...,...,...,...,...
2260650,3084038,4015,"This Sceptred Isle, Vol. 10: The Age of Victor...","[history, historical fiction, biography, non-f...","The award-winning story of Britain, from the a..."
2260651,26168430,2448,Sherlock Holmes and the July Crisis,"[mystery, thriller, crime, children, fiction]",Sir Arthur Conan Doyle is brought back to life...
2260652,2342551,82312,The Children's Classic Poetry Collection,"[poetry, children, young-adult]","Gathers poems by William Blake, Emily Bronte, ..."
2260653,22017381,7789809,"101 Nights: Volume One (101 Nights, #1-3)","[romance, mystery, thriller, crime]","Volume One contains: ""Claimed,"" ""Tainted,"" and..."


In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import torch

# 예시 데이터 준비
# book_id, title, description
book_emb_dict = np.load("book_emb_dict.npy", allow_pickle=True).item()
book_emb_matrix = np.array(list(book_emb_dict.values()))

DATA_PATH = './data/user_book_sequence.parquet'
user_histories = pd.read_parquet(DATA_PATH)

# # 차원 축소 (PCA)
# pca = PCA(n_components=20)
# book_emb_reduced = pca.fit_transform(book_emb_matrix)
book_emb_reduced = book_emb_matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# ---------- 1. Elbow Method: inertia 계산 ----------
k_range = range(2, 21)
sse = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(book_emb_reduced)
    sse.append(kmeans.inertia_)
    print(f"{k}-means 완료!..")

# ---------- 2. Elbow Plot ----------
plt.figure(figsize=(6,4))
plt.plot(k_range, sse, 'o-')
plt.title("Elbow Method")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("SSE (Inertia)")
plt.show()

In [ ]:
# 클러스터링
# 엘보우로 후보 k 3~4개 고르고
# 그 중 실루엣 점수가 가장 높은 k 선택 (실루엣 계산 빡셈)
# book_clusters
# cluster_to_books
books = df.copy()

num_clusters = 5
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(book_emb_reduced)

books['cluster'] = cluster_labels
book_clusters = {books.iloc[i]['book_id']: cluster_labels[i] for i in range(len(books))}

# cluster → book list
cluster_to_books = {}
for i in range(num_clusters):
    cluster_to_books[i] = books[books['cluster']==i]['book_id'].tolist()

print("Books with clusters:")
print(books[['book_id','title','cluster']])

In [14]:
import pyarrow as pa
import pyarrow.parquet as pq

# df_merged 전체를 Parquet으로 저장
out_path = "book_meta.parquet"
table = pa.Table.from_pandas(df_merged)
pq.write_table(table, out_path, compression='snappy')

print("✅ df_merged 전체가 book_meta.parquet로 저장되었습니다.")

✅ df_merged 전체가 book_meta.parquet로 저장되었습니다.
